In [5]:
from langchain_core.runnables import RunnableLambda, RunnableBranch, RunnableParallel, branch
from pydantic import BaseModel, Field

from  langchain_openai import ChatOpenAI
import os
from dotenv import load_dotenv
from openai import OpenAI
import httpx

from langchain_core.messages import HumanMessage, SystemMessage, AIMessage
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser
from typing import Literal, TypedDict
import uuid
import base64
import ssl
from langgraph.graph import StateGraph, START, END
from IPython.display import Image,display, Markdown



# This function will load all the variables from the .env file and will
# make them available in the os.environ dictionary (env variables)
load_dotenv()
ssl_cert_file=""
client= httpx.Client(
        verify=ssl.create_default_context(cafile=ssl_cert_file)
    )

client_id=os.environ.get('LLM_CLIENT_ID')
client_secret=os.environ.get('LLM_CLIENT_SECRET')



auth_string = f"{client_id}:{client_secret}"
auth_header = f"Basic {base64.b64encode(auth_string.encode()).decode()}"

llm_openai = ChatOpenAI()
class graph_schema(BaseModel):
    topic: str = Field(description="The topic of the graph")
    post: str = Field(description="The LinkedIn post content")
    curated_post: str = Field(description="The curated LinkedIn post content")

def create_post(state:graph_schema)->dict:
        # Extract the topic from the state
        topic = state.topic

        # Passing the topic to the LLM to generate a LinkedIn post
        post = llm_openai.invoke(f"Write a LinkedIn post about {topic}").content

        # Return only the field being updated
        return {"post": post}

def curated_post(state:graph_schema)->dict:
        # Extract the topic from the state
        topic = state.topic

        # Passing the topic to the LLM to generate a LinkedIn post
        post = llm_openai.invoke(f"Write a insta post in gen Z {topic}").content

        # Return only the field being updated
        return {"curated_post": post}

def beautify(state:graph_schema)-> dict:
    return {"linkedin": state.post, "instagram": state.curated_post}

graph=StateGraph(graph_schema)

graph.add_node("create_post",create_post)
graph.add_node("curated_post",curated_post)
graph.add_node("beautify",beautify)

graph.add_edge(START,"create_post")
graph.add_edge(START,"curated_post")
graph.add_edge("create_post","beautify")
graph.add_edge("curated_post","beautify")
graph.add_edge("beautify",END)

f_graph=graph.compile()

# Display Mermaid text instead of trying to render as PNG
print(f_graph.get_graph().draw_mermaid())

result=f_graph.invoke({"topic": "The importance of data privacy in the digital age",
     "post": "",
     "curated_post": ""})

print(result)

---
config:
  flowchart:
    curve: linear
---
graph TD;
	__start__([<p>__start__</p>]):::first
	create_post(create_post)
	curated_post(curated_post)
	beautify(beautify)
	__end__([<p>__end__</p>]):::last
	__start__ --> create_post;
	__start__ --> curated_post;
	create_post --> beautify;
	curated_post --> beautify;
	beautify --> __end__;
	classDef default fill:#f2f0ff,line-height:1.2
	classDef first fill-opacity:0
	classDef last fill:#bfb6fc

{'topic': 'The importance of data privacy in the digital age', 'post': '🔐 **Data Privacy: The Non‑Negotiable Competitive Edge in the Digital Age** 🔐  \n\nWe live in a world where every click, swipe, and transaction leaves a digital footprint. As businesses accelerate their digital transformation, **data privacy isn’t just a compliance checkbox—it’s a strategic differentiator**. Here’s why protecting personal information should sit at the top of every leader’s agenda:\n\n1️⃣ **Trust = Loyalty** – Consumers are increasingly savvy about how their data